# 02 · Campaign — linear + macrocyclic peptide design vs MDM2

**Standard slot:** *design campaign.* **For Project 09 this is the core:** run **both** peptide
modalities against the p53 cleft and assemble their pools (D2):
- **Linear peptides** (BoltzGen peptide-anything / EvoBind2) — vary **length**.
- **Macrocycles** (BoltzGen macrocycle-anything / EvoBind2 cyclic) — vary **length + cyclization**.

Then score every design with **AF2/Boltz-2** (`pae_interaction` is the key interface metric) and add
the **Boltz-2 affinity score (relative ranking only — NOT a K_D)**.

> **Compute honesty:** peptides are small — AF2/Boltz-2 scoring and Boltz-2 affinity on small inputs
> run on a **free T4**. A **full macrocycle campaign prefers Colab Pro**. The cells below run on the
> deterministic **mock** backend so the plumbing executes anywhere; the real calls + compute notes are
> shown alongside. Run `00_setup.ipynb` first.

## Setup paths

In [ ]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## Version-verify the pinned upstreams (tools change — BoltzGen especially!)

The peptide tools live in fast-moving upstream repos, and the **BoltzGen public release is moving** —
**verify it explicitly**. **Pin commits/releases** and **verify the URLs still exist** before relying
on them (`requests.head`; a non-200 means it moved — update the pin and log it). The check needs no GPU.

In [ ]:
import requests

# Pinned upstreams (pin a COMMIT/tag/release in your repo — these change; record the exact pin):
#   BoltzGen   <VERIFY the current public release URL>   # peptide-anything / macrocycle-anything — MOVING; verify + pin
#   EvoBind2   https://github.com/patrickbryant1/EvoBind  # cyclic/linear peptide binder design; pin <commit>
#   Boltz      https://github.com/jwohlwend/boltz          # Boltz-2 structure + affinity; pin <version>
#   ColabFold  https://github.com/sokrypton/ColabFold      # AF2 / AF2-Multimer pAE; pin <commit>
PINNED = {
    # BoltzGen: replace with the verified current public release URL once confirmed (see MANUAL.md §2).
    "BoltzGen_repo_candidate": "https://github.com/jwohlwend/boltz",  # PLACEHOLDER — VERIFY the real BoltzGen release URL
    "EvoBind2":  "https://github.com/patrickbryant1/EvoBind",
    "Boltz":     "https://github.com/jwohlwend/boltz",
    "ColabFold": "https://github.com/sokrypton/ColabFold",
}
for name, url in PINNED.items():
    try:
        r = requests.head(url, allow_redirects=True, timeout=15)
        print(f"  [{r.status_code}] {name:24s} {url}")
    except Exception as e:  # noqa: BLE001
        print(f"  [ERR] {name:24s} {url}  ({e})")
print("\nNon-200 / error => the upstream moved; update the pin in env/requirements.txt and log it.")
print("BoltzGen especially: VERIFY the current public release URL before relying on it (it is moving).")

## 1 · Define the campaign

Same target + cleft as notebook 01. Set honest campaign sizes and the length/cyclization sweep; the
cells run on `mock` so they execute anywhere. On Colab switch `TOOL_*` to the real backends — and the
**macrocycle arm prefers Pro** (shrink it on a T4).

In [ ]:
import peptide_tools as pt
import pandas as pd

TARGET = "MDM2"
CLEFT = pt.parse_cleft("A54,A67,A73,A93,A100")   # EXAMPLE — replace with your verified p53-cleft residues

# Length / cyclization sweep (catalog: vary length + constraint). Small mock counts here for a fast dry
# run; scale up with the real backend (a few hundred per modality on Colab; macrocycle arm on Pro).
LINEAR_LENGTHS = [8, 12, 16, 20]     # linear peptide lengths to sweep
CYCLIC_LENGTHS = [8, 11, 14]         # macrocycle lengths to sweep
N_PER_LENGTH   = 15                  # designs per length bucket (mock); scale up on Colab

TOOL_DESIGN = "mock"   # -> "boltzgen" / "evobind2" on Colab
TOOL_SCORE  = "mock"   # -> "boltz" (Boltz-2: pAE + relative affinity) / "af2" (ColabFold pAE) on Colab

print(f"linear lengths   : {LINEAR_LENGTHS}")
print(f"cyclic lengths   : {CYCLIC_LENGTHS}")
print(f"n per length     : {N_PER_LENGTH}   design tool={TOOL_DESIGN}  score tool={TOOL_SCORE}")
print("cleft            :", CLEFT)

## 2 · Modality #1 — linear peptide campaign (length sweep)

Linear peptides across the length sweep. On Colab this is BoltzGen peptide-anything and/or EvoBind2;
we re-score with AF2/Boltz-2 so the modality comparison is apples-to-apples. The `mock` backend returns
deterministic `SYNTHETIC` designs.

In [ ]:
# Real call (Colab): pt.design_peptide(TARGET, CLEFT, length=L, cyclic=False, n=..., tool="boltzgen")
#   or tool="evobind2". See MANUAL.md §2 / scripts/peptide_tools.py TODOs.
linear = []
for L in LINEAR_LENGTHS:
    batch = pt.design_peptide(TARGET, CLEFT, length=L, cyclic=False, n=N_PER_LENGTH, tool=TOOL_DESIGN)
    linear.extend(batch)
pt.score_designs(linear, tool=TOOL_SCORE)     # AF2/Boltz-2 -> pae_interaction, plddt, scrmsd, sc, boltz rank
print(f"linear pool: {len(linear)} designs across lengths {LINEAR_LENGTHS} (tool={TOOL_DESIGN}; SYNTHETIC if mock)")
print("example:", linear[0].design_id, "len=", linear[0].length, "pae=", linear[0].pae_interaction)

## 3 · Modality #2 — macrocyclic campaign (length + cyclization sweep)

Macrocycles across the length sweep with the **cyclic** constraint (head-to-tail / side-chain). On
Colab this is BoltzGen macrocycle-anything / EvoBind2 cyclic — and **prefers Colab Pro**. The `mock`
backend stands in for the whole chain.

In [ ]:
# Real call (Colab, Pro for the campaign): pt.design_peptide(TARGET, CLEFT, length=L, cyclic=True,
#   n=..., tool="boltzgen") or tool="evobind2". The macrocycle arm is the heavier one.
macro = []
for L in CYCLIC_LENGTHS:
    batch = pt.design_peptide(TARGET, CLEFT, length=L, cyclic=True, n=N_PER_LENGTH, tool=TOOL_DESIGN)
    macro.extend(batch)
pt.score_designs(macro, tool=TOOL_SCORE)
print(f"macrocycle pool: {len(macro)} designs across lengths {CYCLIC_LENGTHS} (tool={TOOL_DESIGN}; SYNTHETIC if mock)")
print("example:", macro[0].design_id, "len=", macro[0].length, "cyclic=", macro[0].cyclic,
      "pae=", macro[0].pae_interaction)

## 4 · Mini-protein FOIL (for the peptide-vs-protein modality comparison)

To compare *modalities* (notebook 04), we also generate a **mini-protein binder foil** against the
same cleft — a Project-06-style mini-binder. Here it is mocked; on Colab the real path defers to the
Project-06 binder workflow (BindCraft / RFdiffusion-binder + ProteinMPNN, **A100**). Generate it at a
comparable scale so the comparison is fair.

In [ ]:
# Real path (Colab, A100): the Project-06 binder workflow against the same MDM2 cleft.
foil = pt.design_miniprotein_foil(TARGET, CLEFT, n=2 * N_PER_LENGTH, tool="mock")
pt.score_designs(foil, tool=TOOL_SCORE)
print(f"mini-protein foil: {len(foil)} designs (mock; real = Project-06 workflow on A100)")
print("example:", foil[0].design_id, "len=", foil[0].length, "pae=", foil[0].pae_interaction)

## 5 · Assemble + persist the pools

Write one tidy CSV per modality (plus a combined one). These feed notebook 03 (the shared filter).
We add an EXAMPLE physics column (`rosetta_dG`) here so the binder physics layer has something to act
on in the dry run — on Colab replace with a real interface-energy estimate; for `mock` it is SYNTHETIC.

In [ ]:
import pandas as pd

def pool_to_df(designs):
    rows = []
    for d in designs:
        # In the mock dry run we attach an EXAMPLE_DATA interface energy so Layer 3 (physics) is
        # exercised. On Colab, replace with a real interface-energy / solubility estimate.
        rdg = -45.0 + (pt._hashints("dG", d.design_id) % 40)   # SYNTHETIC, range ~ -45..-6 REU
        rows.append(dict(
            design_id=d.design_id, modality=d.modality, tool=d.tool, target=d.target,
            length=d.length, cyclic=d.cyclic, sequence=d.sequence,
            plddt=d.plddt, pae_interaction=d.pae_interaction, scrmsd=d.scrmsd,
            shape_complementarity=d.shape_complementarity,
            boltz_affinity_score=d.boltz_affinity_score,   # RELATIVE RANK, never a K_D
            rosetta_dG=round(float(rdg), 2), solubility=0.3,
            contact_residues=",".join(d.contact_residues),
            cleft_overlap=pt.cleft_overlap(d.contact_residues, d.cleft),
            synthetic=d.synthetic,
        ))
    return pd.DataFrame(rows)

df_lin  = pool_to_df(linear); df_lin.to_csv("results/linear_designs.csv", index=False)
df_cyc  = pool_to_df(macro);  df_cyc.to_csv("results/macrocycle_designs.csv", index=False)
df_foil = pool_to_df(foil);   df_foil.to_csv("results/miniprotein_foil_designs.csv", index=False)
combined = pd.concat([df_lin, df_cyc, df_foil], ignore_index=True)
combined.to_csv("results/all_designs.csv", index=False)

print("wrote results/linear_designs.csv          ", df_lin.shape)
print("wrote results/macrocycle_designs.csv      ", df_cyc.shape)
print("wrote results/miniprotein_foil_designs.csv", df_foil.shape)
print("wrote results/all_designs.csv             ", combined.shape)
print("\nALL numbers are SYNTHETIC in the mock dry run (EXAMPLE_DATA). boltz_affinity_score is a")
print("RELATIVE RANK, never a K_D. Never report any of this as real results.")
combined.head(4)

## D2 checklist
- [ ] Linear pool generated across a **length** sweep (BoltzGen peptide-anything / EvoBind2 on Colab).
- [ ] Macrocycle pool generated across a **length + cyclization** sweep (macrocycle arm on Pro).
- [ ] Mini-protein **foil** generated (Project-06 workflow on A100) for the modality comparison.
- [ ] Every design scored by AF2/Boltz-2 (`pae_interaction` parsed; Boltz-2 affinity as a *rank*).
- [ ] Pools written to `results/`; design log (every config + seed + tool **commit/release** + path) in `LOG.md`.
- [ ] Version-verify output captured (incl. the BoltzGen-release check); 3–4 page interim report.

**Next:** `03_filter_and_rank.ipynb` — run the **shared** filter on the pools.